# 1. The relation of Sample & lane

#### In this project, worker has 6 duplicates (GSM) and gamergate has 5 duplicates (GSM), each GSM has **single** run (SRR), and each SRR contains two fastqs.

In [1]:
%%bash

# fetch the SRR name of GSM

while IFS= read -r GSM || [[ -n "$GSM" ]]; do # [[ -n "$GSM" ]] 防止文件最后一行没有换行符异常退出
    ffq "$GSM" > "./metadata/${GSM}.json"
done < "./metadata/all_GSM"

[2026-04-30 16:59:32,809]    INFO Parsing GSM GSM4013393
[2026-04-30 16:59:33,468]    INFO Finding supplementary files for GSM GSM4013393
[2026-04-30 16:59:37,532]    INFO No supplementary files found for GSM4013393
[2026-04-30 16:59:38,692]    INFO Getting sample for GSM4013393
[2026-04-30 16:59:40,226]    INFO Parsing sample SRS5226320
[2026-04-30 16:59:41,553] WARNING Failed to parse sample information from ENA XML. Falling back to ENA search...
[2026-04-30 16:59:42,635]    INFO Getting Experiment for SRS5226320
[2026-04-30 16:59:42,635]    INFO Parsing Experiment SRX6666526
[2026-04-30 16:59:42,641] WARNING Failed to parse experiment information from ENA XML. Falling back to ENA search...
[2026-04-30 16:59:43,713] WARNING There is 1 run for SRX6666526
[2026-04-30 16:59:43,713]    INFO Parsing run SRR9917388
[2026-04-30 16:59:48,198]    INFO Parsing GSM GSM4013392
[2026-04-30 16:59:48,559]    INFO Finding supplementary files for GSM GSM4013392
[2026-04-30 16:59:51,259]    INFO No su

In [3]:
import json

def GSM_SRR_relation(GSM_path):
    SRR = []
    with open(GSM_path, "r") as f:
        data = json.load(f)
    gsm = list(data.keys())[0]
    gsm_data = data[gsm]
    # srp = None
    for sample in gsm_data["samples"].values():
        for exp in sample["experiments"].values():
            for run in exp["runs"].values():
                srr = run["accession"]
                SRR.append(srr)
    return gsm, SRR

all_GSM_SRR = {}

with open("./metadata/all_GSM", "r") as f:
    for GSM_name in f:
        GSM_name = GSM_name.rstrip()
        GSM_path = f"./metadata/{GSM_name}.json"
        all_GSM_SRR[GSM_SRR_relation(GSM_path)[0]] = GSM_SRR_relation(GSM_path)[1]

all_GSM_SRR

{'GSM4013393': ['SRR9917388'],
 'GSM4013392': ['SRR9917387'],
 'GSM4013391': ['SRR9917386'],
 'GSM4013390': ['SRR9917385'],
 'GSM4013389': ['SRR9917384'],
 'GSM4013388': ['SRR9917383'],
 'GSM4013387': ['SRR9917382'],
 'GSM4013386': ['SRR9917381'],
 'GSM4013385': ['SRR9917380'],
 'GSM4013384': ['SRR9917379'],
 'GSM4013383': ['SRR9917378']}

In [4]:
with open("./metadata/all_GSM_SRR.json", "w") as f:
    json.dump(all_GSM_SRR, f)

#### Based on the relationship between GSM and SRR, rebulid the Fastq file system (move the SRR fastq file into GSM directory): 

In [5]:
import json

with open("./metadata/all_GSM_SRR.json", "r") as f:
    all_GSM_SRR = json.load(f)

In [7]:
from pathlib import Path

for GSM_name in all_GSM_SRR:
    GSM_dir = Path(f"./fastq/{GSM_name}")
    if not GSM_dir.exists():
        GSM_dir.mkdir(parents=True, exist_ok=True)
    for SRR_name in all_GSM_SRR[GSM_name]:
        # print(SRR_name)
        matched_files = list(Path("./fastq").glob(f"{SRR_name}*.fastq.gz"))
        for SRR_file in matched_files:
            # print(SRR_file)
            target_SRR_file = GSM_dir / SRR_file.name
            SRR_file.rename(target_SRR_file)

#### Rename the fastq file into `<SampleName>_S<SampleNumber>_L00<Lane>_<Read>_001.fastq.gz` format

In [8]:
%%bash

ls "./fastq/GSM4013393"

# SRR9917388.lite.1_1.fastq.gz
# SRR9917388.lite.1_2.fastq.gz

SRR9917388.lite.1_1.fastq.gz
SRR9917388.lite.1_2.fastq.gz


In [9]:
for GSM_name in all_GSM_SRR:
    # print(GSM_name)
    value_list = [int(item[-2:]) for item in all_GSM_SRR[GSM_name]]
    base_value = min(value_list) - 1
    for SRR_name in all_GSM_SRR[GSM_name]:
        lane_value = int(SRR_name[-2:]) - base_value
        origin_name_R1 = Path(f"./fastq/{GSM_name}/{SRR_name}.lite.1_1.fastq.gz")
        origin_name_R2 = Path(f"./fastq/{GSM_name}/{SRR_name}.lite.1_2.fastq.gz")
        neo_name_R1 = Path(f"./fastq/{GSM_name}/{GSM_name}_S1_L00{lane_value}_R1_001.fastq.gz")
        neo_name_R2 = Path(f"./fastq/{GSM_name}/{GSM_name}_S1_L00{lane_value}_R2_001.fastq.gz")
        if origin_name_R1.exists() & origin_name_R2.exists():
            origin_name_R1.rename(neo_name_R1)
            origin_name_R2.rename(neo_name_R2)

# <SampleName>_S<SampleNumber>_L00<Lane>_<Read>_001.fastq.gz

In [10]:
%%bash

ls "./fastq/GSM4013393"

GSM4013393_S1_L001_R1_001.fastq.gz
GSM4013393_S1_L001_R2_001.fastq.gz
